Manual training

In [1]:
import cv2
import numpy as np 
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
import pickle
from skimage.feature import local_binary_pattern
from PIL import Image
import random
import copy
from sklearn.utils import shuffle

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
# from pygist import Gist

%matplotlib inline 

ModuleNotFoundError: No module named 'cv2'

In [ ]:
def PerformanceMetrics(test_labels, results):
    n_classes = int(max(test_labels).astype(int) + 1)

    # Initialize confusion matrix and accuracy
    conf_matrix = np.zeros((n_classes, n_classes))
    accuracy = 0

    # Populate confusion matrix and calculate accuracy
    for index, i in enumerate(test_labels):
        conf_matrix[i, results[index]] += 1
        if results[index] == i:
            accuracy += 1

    accuracy /= len(test_labels)

    # Initialize precision, recall, and F1 arrays
    precision = np.zeros(n_classes)
    recall = np.zeros(n_classes)
    F1 = np.zeros(n_classes)

    # Calculate metrics for each class
    for k in range(n_classes):
        TP = 0
        FP = 0
        FN = 0

        for index, i in enumerate(test_labels):
            if i == k and results[index] == k:
                TP += 1  # True Positive
            elif i != k and results[index] == k:
                FP += 1  # False Positive
            elif i == k and results[index] != k:
                FN += 1  # False Negative

        precision[k] = TP / (TP + FP + 1e-4)  # Precision
        recall[k] = TP / (TP + FN + 1e-4)    # Recall
        F1[k] = 2 * (precision[k] * recall[k]) / (precision[k] + recall[k] + 1e-4)  # F1 Score

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "F1": F1,
        "confusion_matrix": conf_matrix
    }

In [ ]:
def classification_metrics(y_true, y_pred, average='binary'):
    """
    Calculate accuracy, precision, recall, F1-score, and confusion matrix.

    Parameters:
    -----------
    y_true : list or array-like
        Ground truth labels.
    y_pred : list or array-like
        Predicted labels.
    average : str, default='binary'
        Averaging method for multi-class data. Options:
        ['binary', 'micro', 'macro', 'weighted'].
    
    Returns:
    --------
    metrics : dict
        Dictionary containing accuracy, precision, recall, f1, and confusion matrix.
    """
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=average, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }
    return metrics

In [ ]:
classes = {
    "TwoHand_L_Fist_R_Fist": {},
    "TwoHand_L_Flat_R_Flat": {},
    "TwoHand_L_Okay_R_Okay": {},
    "TwoHand_L_Rock_R_Rock": {}
}


dataset_size = 0
for key in classes.keys():
    for filename in os.listdir("TrainingData/" + key):
        if filename.lower().endswith(".jpg"):
            classes[key][filename[:-4]] = {
                "im_rgb": cv2.imread(os.path.join("TrainingData/" + key, filename)),
                "im_gray": cv2.imread(os.path.join("TrainingData/" + key, filename), cv2.IMREAD_GRAYSCALE),
                "im_hsv": cv2.cvtColor(cv2.imread(os.path.join("TrainingData/" + key, filename)), cv2.COLOR_RGB2HSV)
            }
            dataset_size += 1

print(dataset_size)
